# Reach — 100 Demonstrations

In [ ]:
import numpy as np, matplotlib.pyplot as plt, json, random
from pathlib import Path
DATA = Path('/home/shareef/robot_zoo/data/demos/franka_reach')
meta = json.load(open(DATA/'metadata.json'))
print('Task:', meta['task'], '| Episodes:', meta['total_episodes'], '| Success rate:', meta['success_rate'])

In [ ]:
episodes = sorted(DATA.glob('episode_*.npz'))
ep = np.load(episodes[0])
print('Keys:', list(ep.keys()))
print('observation_state:', ep['observation_state'].shape)
print('action:', ep['action'].shape)

In [ ]:
# Joint trajectories for 3 random episodes
sample = random.sample(episodes, 3)
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax, path in zip(axes, sample):
    ep = np.load(path)
    jp = ep['observation_state'][:, :9]
    for j in range(7): ax.plot(jp[:, j], label=f'j{j}', alpha=0.8)
    ax.set_title(f'{path.name} | steps: {len(jp)}')
    ax.set_ylabel('Joint pos (rad)'); ax.legend(fontsize=7, ncol=4); ax.grid(alpha=0.3)
plt.suptitle('Reach — Joint Trajectories', fontsize=13)
plt.tight_layout()
plt.savefig(DATA/'joint_trajectories.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Target positions across all episodes (XYZ scatter)
targets = np.array([np.load(e)['observation_cube_pos'][0] for e in episodes])
fig = plt.figure(figsize=(14, 5))
labels = ['X (forward)', 'Y (side)', 'Z (height)']
colors = ['steelblue', 'coral', 'green']
for i in range(3):
    ax = fig.add_subplot(1, 3, i+1)
    ax.hist(targets[:, i], bins=20, color=colors[i], alpha=0.7, edgecolor='black')
    ax.set_title(f'Target {labels[i]}')
    ax.set_xlabel('Position (m)'); ax.grid(alpha=0.3)
plt.suptitle('Reach Target Distribution — 100 Episodes', fontsize=13)
plt.tight_layout()
plt.savefig(DATA/'target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Dataset statistics
lengths = [len(np.load(e)['reward']) for e in episodes]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_title('Episode Length Distribution'); ax.set_xlabel('Steps'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA/'dataset_stats.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Mean length: {np.mean(lengths):.1f} | Total steps: {sum(lengths):,}')

In [ ]:
# Action distribution
all_actions = np.concatenate([np.load(e)['action'] for e in episodes])
fig, axes = plt.subplots(1, 7, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.hist(all_actions[:, i], bins=30, alpha=0.7, color='purple', edgecolor='black')
    ax.set_title(f'arm_j{i}'); ax.grid(alpha=0.3)
plt.suptitle('Action Distribution — Reach', fontsize=13)
plt.tight_layout()
plt.savefig(DATA/'action_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Total timesteps: {len(all_actions):,}')

In [ ]:
print('='*45)
print('WEEK 3 COMPLETE — Dataset Summary')
print('='*45)
for task, path in [('pick_cube', '/home/shareef/robot_zoo/data/demos/franka_pick_cube'),
                   ('open_drawer', '/home/shareef/robot_zoo/data/demos/franka_open_drawer'),
                   ('reach', '/home/shareef/robot_zoo/data/demos/franka_reach')]:
    eps = list(Path(path).glob('episode_*.npz'))
    total_steps = sum(len(np.load(e)['reward']) for e in eps)
    print(f'  {task:15s}: {len(eps):3d} episodes | {total_steps:6,} steps')
print('='*45)